# PCC GPU compatibility probe
Environment-only probe. No dataset, manifest, or scientific execution.

In [ ]:
from pathlib import Path
import json, platform, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
import torch
assert torch.cuda.is_available(), 'GPU_REQUIRED=true but CUDA is unavailable'
device = torch.device('cuda')
properties = torch.cuda.get_device_properties(0)
x = torch.randn(2, 2, 32, 32, device=device, requires_grad=True)
model = torch.nn.Sequential(torch.nn.Conv2d(2, 8, 3, padding=1), torch.nn.BatchNorm2d(8), torch.nn.ReLU(), torch.nn.Conv2d(8, 1, 1)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer.zero_grad(); y = model(x); loss = y.square().mean(); loss.backward(); optimizer.step(); torch.cuda.synchronize()
result = {'status':'PASS','gpu_required':True,'torch':torch.__version__,'torch_cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),'gpu_name':properties.name,'compute_capability':f'{properties.major}.{properties.minor}','python':platform.python_version(),'output_shape':list(y.shape),'backward':True,'optimizer_step':True}
out = Path('/kaggle/working/gpu_probe_result.json'); out.write_text(json.dumps(result, indent=2)); print(json.dumps(result, indent=2))
